# 03 Build Complexity Metrics

Aggregate Frankfurt Silver flight states into interpretable 5-minute grid complexity metrics and Gold reporting tables.

Primary targets:

- `adsb_airspace_eddf.gld_airspace.grid_complexity_5m`
- `adsb_airspace_eddf.gld_airspace.complexity_hotspots`
- `adsb_airspace_eddf.gld_airspace.complexity_trend_15m`
- `adsb_airspace_eddf.obs.pipeline_run_log`

## Complexity Contract

This notebook computes one Gold run per selected Silver `run_id`.

- Grain: `grid_id x 5-minute window`
- Features: `aircraft_count`, `heading_dispersion`, `altitude_dispersion`, `speed_dispersion`
- Score: weighted z-score blend using `configs/pipeline_config.yaml`
- Hotspots: grid ranking by average complexity score with a `percentile_approx(..., 0.9)` threshold for high-complexity windows
- Trend: 15-minute rollup for dashboard-friendly summaries

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import sys
import yaml

from pyspark.sql import Window
from pyspark.sql import functions as F

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

UTC = timezone.utc
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

with (repo_root / "configs" / "region_config.yaml").open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)

with (repo_root / "configs" / "pipeline_config.yaml").open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)

def parse_bool(raw_value: str) -> bool:
    return raw_value.strip().lower() == "true"

def utc_now_naive() -> datetime:
    return datetime.now(UTC).replace(tzinfo=None)

def normalize_scalar(value):
    if value is None:
        return None
    if hasattr(value, "to_pydatetime"):
        return value.to_pydatetime()
    return value

def append_row_to_table(*, target_table: str, payload: dict) -> None:
    target_schema = spark.table(target_table).schema
    row = {
        field.name: normalize_scalar(payload.get(field.name))
        for field in target_schema
    }
    spark.createDataFrame([row], schema=target_schema).write.mode("append").saveAsTable(target_table)

def zscore_expr(column_name: str, *, mean_value, std_value):
    mean_number = 0.0 if mean_value is None else float(mean_value)
    if std_value is None or abs(float(std_value)) < 1e-9:
        return F.lit(0.0)
    return (F.col(column_name) - F.lit(mean_number)) / F.lit(float(std_value))

default_catalog = pipeline_config.get("catalog_name", "adsb_airspace_eddf")
default_source_run_id = ""
default_overwrite_source_run = "true"

catalog = default_catalog
source_run_id_raw = default_source_run_id
overwrite_source_run_raw = default_overwrite_source_run

if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    dbutils.widgets.text("source_run_id", default_source_run_id)
    dbutils.widgets.dropdown("overwrite_source_run", default_overwrite_source_run, ["true", "false"])

    catalog = dbutils.widgets.get("catalog").strip() or default_catalog
    source_run_id_raw = dbutils.widgets.get("source_run_id").strip()
    overwrite_source_run_raw = dbutils.widgets.get("overwrite_source_run")

focus_airport = region_config["focus_airport"]
trend_window_minutes = int(pipeline_config["time_windows"]["trend_window_minutes"])
complexity_weights = pipeline_config["complexity_weights"]
overwrite_source_run = parse_bool(overwrite_source_run_raw)

silver_table = f"{catalog}.slv_airspace.flight_states_clean"
grid_complexity_table = f"{catalog}.gld_airspace.grid_complexity_5m"
hotspots_table = f"{catalog}.gld_airspace.complexity_hotspots"
trend_table = f"{catalog}.gld_airspace.complexity_trend_15m"
pipeline_log_table = f"{catalog}.obs.pipeline_run_log"
source_run_id_input = source_run_id_raw.strip()


In [ ]:
if source_run_id_input:
    selected_source_run_id = source_run_id_input
else:
    latest_success_row = spark.sql(
        f"""
        SELECT run_id
        FROM {catalog}.obs.pipeline_run_log
        WHERE pipeline_name = '02_clean_and_prepare_states'
          AND status = 'success'
        ORDER BY completed_at DESC
        LIMIT 1
        """
    ).first()
    if latest_success_row is None:
        raise ValueError("No successful Silver run found. Run 02_clean_and_prepare_states.ipynb first.")
    selected_source_run_id = latest_success_row["run_id"]

silver_df = spark.table(silver_table).where(F.col("run_id") == F.lit(selected_source_run_id))
source_row_count = silver_df.count()
if source_row_count == 0:
    raise ValueError(f"No Silver rows found for run_id={selected_source_run_id}.")

source_summary_row = silver_df.agg(
    F.countDistinct("time_window_5m").alias("window_count"),
    F.countDistinct("icao24").alias("aircraft_count"),
    F.countDistinct("grid_id").alias("grid_count"),
    F.min("event_time").alias("min_event_time"),
    F.max("event_time").alias("max_event_time")
).first()

run_plan = {
    "catalog": catalog,
    "focus_airport": focus_airport,
    "source_run_id": selected_source_run_id,
    "silver_table": silver_table,
    "grid_complexity_table": grid_complexity_table,
    "hotspots_table": hotspots_table,
    "trend_table": trend_table,
    "trend_window_minutes": trend_window_minutes,
    "overwrite_source_run": overwrite_source_run,
    "source_row_count": source_row_count,
    "source_window_count": int(source_summary_row["window_count"]),
    "source_aircraft_count": int(source_summary_row["aircraft_count"]),
    "source_grid_count": int(source_summary_row["grid_count"]),
    "min_event_time": str(source_summary_row["min_event_time"]),
    "max_event_time": str(source_summary_row["max_event_time"]),
    "complexity_weights": {key: float(value) for key, value in complexity_weights.items()},
}

run_plan


In [ ]:
grid_metrics_base_df = (
    silver_df
    .groupBy("time_window_5m", "grid_id")
    .agg(
        F.countDistinct("icao24").alias("aircraft_count"),
        F.avg(F.cos(F.radians(F.col("heading_deg")))).alias("mean_heading_cos"),
        F.avg(F.sin(F.radians(F.col("heading_deg")))).alias("mean_heading_sin"),
        F.count(F.col("heading_deg")).alias("heading_obs_count"),
        F.stddev_samp("altitude_ft").alias("altitude_dispersion_raw"),
        F.stddev_samp("speed_kt").alias("speed_dispersion_raw")
    )
    .withColumn(
        "heading_dispersion",
        F.when(
            F.col("heading_obs_count") <= 1,
            F.lit(0.0),
        ).otherwise(
            F.lit(1.0) - F.sqrt(
                F.pow(F.coalesce(F.col("mean_heading_cos"), F.lit(0.0)), F.lit(2.0)) +
                F.pow(F.coalesce(F.col("mean_heading_sin"), F.lit(0.0)), F.lit(2.0))
            )
        )
    )
    .withColumn("altitude_dispersion", F.coalesce(F.col("altitude_dispersion_raw"), F.lit(0.0)))
    .withColumn("speed_dispersion", F.coalesce(F.col("speed_dispersion_raw"), F.lit(0.0)))
)

metric_stats_row = grid_metrics_base_df.agg(
    F.avg("aircraft_count").alias("aircraft_count_mean"),
    F.stddev_samp("aircraft_count").alias("aircraft_count_std"),
    F.avg("heading_dispersion").alias("heading_dispersion_mean"),
    F.stddev_samp("heading_dispersion").alias("heading_dispersion_std"),
    F.avg("altitude_dispersion").alias("altitude_dispersion_mean"),
    F.stddev_samp("altitude_dispersion").alias("altitude_dispersion_std"),
    F.avg("speed_dispersion").alias("speed_dispersion_mean"),
    F.stddev_samp("speed_dispersion").alias("speed_dispersion_std")
).first()

aircraft_weight = float(complexity_weights["aircraft_count"])
heading_weight = float(complexity_weights["heading_dispersion"])
altitude_weight = float(complexity_weights["altitude_dispersion"])
speed_weight = float(complexity_weights["speed_dispersion"])

grid_metrics_df = (
    grid_metrics_base_df
    .withColumn("z_aircraft_count", zscore_expr("aircraft_count", mean_value=metric_stats_row["aircraft_count_mean"], std_value=metric_stats_row["aircraft_count_std"]))
    .withColumn("z_heading_dispersion", zscore_expr("heading_dispersion", mean_value=metric_stats_row["heading_dispersion_mean"], std_value=metric_stats_row["heading_dispersion_std"]))
    .withColumn("z_altitude_dispersion", zscore_expr("altitude_dispersion", mean_value=metric_stats_row["altitude_dispersion_mean"], std_value=metric_stats_row["altitude_dispersion_std"]))
    .withColumn("z_speed_dispersion", zscore_expr("speed_dispersion", mean_value=metric_stats_row["speed_dispersion_mean"], std_value=metric_stats_row["speed_dispersion_std"]))
    .withColumn(
        "complexity_score_raw",
        aircraft_weight * F.col("z_aircraft_count") +
        heading_weight * F.col("z_heading_dispersion") +
        altitude_weight * F.col("z_altitude_dispersion") +
        speed_weight * F.col("z_speed_dispersion")
    )
    .select(
        F.col("time_window_5m").alias("window_start"),
        F.col("grid_id"),
        F.col("aircraft_count").cast("long").alias("aircraft_count"),
        F.round(F.col("heading_dispersion"), 6).alias("heading_dispersion"),
        F.round(F.col("altitude_dispersion"), 6).alias("altitude_dispersion"),
        F.round(F.col("speed_dispersion"), 6).alias("speed_dispersion"),
        F.round(F.col("complexity_score_raw"), 6).alias("complexity_score"),
        F.current_timestamp().alias("computed_at"),
        F.lit(selected_source_run_id).alias("run_id")
    )
)

grid_quality_row = grid_metrics_df.agg(
    F.count("*").alias("grid_window_count"),
    F.countDistinct("grid_id").alias("grid_count"),
    F.countDistinct("window_start").alias("window_count"),
    F.min("complexity_score").alias("min_complexity_score"),
    F.max("complexity_score").alias("max_complexity_score")
).first()

if int(grid_quality_row["grid_window_count"]) == 0:
    raise ValueError("Complexity aggregation produced no rows. Check the Silver source run.")

hotspot_threshold_row = grid_metrics_df.select(
    F.expr("percentile_approx(complexity_score, 0.9)").alias("hotspot_threshold")
).first()
hotspot_threshold = float(hotspot_threshold_row["hotspot_threshold"] or 0.0)

ranking_window = Window.orderBy(
    F.col("avg_complexity_score").desc(),
    F.col("peak_complexity_score").desc(),
    F.col("grid_id").asc(),
)

hotspots_df = (
    grid_metrics_df
    .groupBy("grid_id")
    .agg(
        F.round(F.avg("complexity_score"), 6).alias("avg_complexity_score"),
        F.round(F.max("complexity_score"), 6).alias("peak_complexity_score"),
        F.sum(F.when(F.col("complexity_score") >= F.lit(hotspot_threshold), F.lit(1)).otherwise(F.lit(0))).cast("long").alias("num_high_complexity_windows")
    )
    .withColumn("ranking", F.dense_rank().over(ranking_window).cast("int"))
    .withColumn("computed_at", F.current_timestamp())
    .withColumn("run_id", F.lit(selected_source_run_id))
    .select(
        "grid_id",
        "avg_complexity_score",
        "peak_complexity_score",
        "num_high_complexity_windows",
        "ranking",
        "computed_at",
        "run_id",
    )
)

trend_window_seconds = trend_window_minutes * 60
grid_trend_df = grid_metrics_df.withColumn(
    "window_start_15m",
    F.to_timestamp(F.from_unixtime(F.floor(F.unix_timestamp("window_start") / F.lit(trend_window_seconds)) * F.lit(trend_window_seconds)))
)

aircraft_trend_df = silver_df.withColumn(
    "window_start_15m",
    F.to_timestamp(F.from_unixtime(F.floor(F.unix_timestamp("time_window_5m") / F.lit(trend_window_seconds)) * F.lit(trend_window_seconds)))
).groupBy("window_start_15m").agg(
    F.countDistinct("icao24").alias("active_aircraft_count")
)

trend_df = (
    grid_trend_df
    .groupBy("window_start_15m")
    .agg(
        F.round(F.avg("complexity_score"), 6).alias("avg_complexity_score"),
        F.round(F.max("complexity_score"), 6).alias("peak_complexity_score"),
        F.countDistinct("grid_id").alias("active_grid_count")
    )
    .join(aircraft_trend_df, on="window_start_15m", how="left")
    .withColumn("computed_at", F.current_timestamp())
    .withColumn("run_id", F.lit(selected_source_run_id))
    .select(
        F.col("window_start_15m").alias("window_start"),
        "avg_complexity_score",
        "peak_complexity_score",
        F.col("active_grid_count").cast("long").alias("active_grid_count"),
        F.col("active_aircraft_count").cast("long").alias("active_aircraft_count"),
        "computed_at",
        "run_id",
    )
)

hotspots_count = hotspots_df.count()
trend_count = trend_df.count()

metric_summary = {
    "source_run_id": selected_source_run_id,
    "source_row_count": source_row_count,
    "grid_window_count": int(grid_quality_row["grid_window_count"]),
    "grid_count": int(grid_quality_row["grid_count"]),
    "window_count": int(grid_quality_row["window_count"]),
    "hotspots_count": int(hotspots_count),
    "trend_count": int(trend_count),
    "hotspot_threshold": hotspot_threshold,
    "min_complexity_score": float(grid_quality_row["min_complexity_score"]),
    "max_complexity_score": float(grid_quality_row["max_complexity_score"]),
}

metric_summary


In [ ]:
pipeline_started_at = utc_now_naive()
pipeline_status = "success"
pipeline_error_message = None

try:
    if overwrite_source_run:
        spark.sql(f"DELETE FROM {grid_complexity_table} WHERE run_id = '{selected_source_run_id}'")
        spark.sql(f"DELETE FROM {hotspots_table} WHERE run_id = '{selected_source_run_id}'")
        spark.sql(f"DELETE FROM {trend_table} WHERE run_id = '{selected_source_run_id}'")

    grid_metrics_df.write.mode("append").saveAsTable(grid_complexity_table)
    hotspots_df.write.mode("append").saveAsTable(hotspots_table)
    trend_df.write.mode("append").saveAsTable(trend_table)
except Exception as exc:
    pipeline_status = "failed"
    pipeline_error_message = f"{type(exc).__name__}: {exc}"
    raise
finally:
    append_row_to_table(
        target_table=pipeline_log_table,
        payload={
            "run_id": selected_source_run_id,
            "pipeline_name": "03_build_complexity_metrics",
            "layer": "gold",
            "target_table": grid_complexity_table,
            "status": pipeline_status,
            "rows_read": source_row_count,
            "rows_written": (
                metric_summary["grid_window_count"] + metric_summary["hotspots_count"] + metric_summary["trend_count"]
                if pipeline_status == "success"
                else 0
            ),
            "started_at": pipeline_started_at,
            "completed_at": utc_now_naive(),
            "error_message": pipeline_error_message,
            "metadata_json": json.dumps(
                {
                    "source_run_id": selected_source_run_id,
                    "focus_airport": focus_airport,
                    "trend_window_minutes": trend_window_minutes,
                    "overwrite_source_run": overwrite_source_run,
                    "target_tables": [grid_complexity_table, hotspots_table, trend_table],
                    "complexity_weights": {key: float(value) for key, value in complexity_weights.items()},
                    "hotspot_threshold": hotspot_threshold,
                },
                sort_keys=True,
                default=str,
            ),
        },
    )

final_summary = {
    "status": pipeline_status,
    "source_run_id": selected_source_run_id,
    "source_row_count": source_row_count,
    "grid_complexity_table": grid_complexity_table,
    "hotspots_table": hotspots_table,
    "trend_table": trend_table,
    "grid_window_count": metric_summary["grid_window_count"],
    "hotspots_count": metric_summary["hotspots_count"],
    "trend_count": metric_summary["trend_count"],
    "hotspot_threshold": metric_summary["hotspot_threshold"],
    "min_complexity_score": metric_summary["min_complexity_score"],
    "max_complexity_score": metric_summary["max_complexity_score"],
}

final_summary
